# 05 - Kafka → HDFS
Consumer 2 : stocke les messages Kafka dans HDFS pour le traitement par lot

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# JARs Kafka montes dans /opt/spark-kafka-jars/
KAFKA_JARS = ','.join([
    '/opt/spark-kafka-jars/spark-sql-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/spark-token-provider-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/kafka-clients-3.4.1.jar',
    '/opt/spark-kafka-jars/commons-pool2-2.11.1.jar'
])

spark = SparkSession.builder \
    .appName('05_Kafka_to_HDFS') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.jars', KAFKA_JARS) \
    .getOrCreate()

print('Spark connecte')

Spark connecte


In [2]:
# Schema des messages Kafka
trip_schema = StructType([
    StructField('date_id', DoubleType()),
    StructField('pickup_location_id', IntegerType()),
    StructField('dropoff_location_id', IntegerType()),
    StructField('weather_id', DoubleType()),
    StructField('trip_distance', DoubleType()),
    StructField('total_amount', DoubleType())
])

# Lire le flux Kafka
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'nyc-taxi-trips') \
    .option('startingOffsets', 'earliest') \
    .load()

# Parser les messages JSON
trips_parsed = kafka_df \
    .select(from_json(col('value').cast('string'), trip_schema).alias('data')) \
    .select('data.*')

print('Flux Kafka connecte')

Flux Kafka connecte


In [3]:
# Ecrire le flux dans HDFS en Parquet (micro-batches)
HDFS_OUTPUT = 'hdfs://namenode:9000/projet/streaming/kafka_trips'
CHECKPOINT = 'hdfs://namenode:9000/projet/streaming/_checkpoint'

query = trips_parsed.writeStream \
    .outputMode('append') \
    .format('parquet') \
    .option('path', HDFS_OUTPUT) \
    .option('checkpointLocation', CHECKPOINT) \
    .trigger(processingTime='10 seconds') \
    .start()

print(f'Kafka → HDFS demarre')
print(f'Les donnees arrivent dans : {HDFS_OUTPUT}')
print(f'Checkpoint : {CHECKPOINT}')

Kafka → HDFS demarre
Les donnees arrivent dans : hdfs://namenode:9000/projet/streaming/kafka_trips
Checkpoint : hdfs://namenode:9000/projet/streaming/_checkpoint


In [6]:
# Verifier les donnees ecrites dans HDFS
import time
time.sleep(15)  # Attendre quelques micro-batches

try:
    hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
    print(f'Lignes stockees dans HDFS : {hdfs_trips.count()}')
    hdfs_trips.show(5)
except Exception as e:
    print(f'Pas encore de donnees : {e}')

Lignes stockees dans HDFS : 213941
+-------+------------------+-------------------+----------+-------------+------------+
|date_id|pickup_location_id|dropoff_location_id|weather_id|trip_distance|total_amount|
+-------+------------------+-------------------+----------+-------------+------------+
|  327.0|               168|                159|      NULL|         1.14|        16.8|
|  257.0|                79|                 41|      NULL|         8.46|       52.44|
|  268.0|                70|                 79|      NULL|          9.7|        69.7|
|  171.0|               235|                139|      NULL|         0.61|        12.5|
|   67.0|               160|                232|      NULL|         1.55|        19.8|
+-------+------------------+-------------------+----------+-------------+------------+
only showing top 5 rows



In [7]:
# Re-executez pour voir le compteur augmenter
hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
print(f'Lignes stockees dans HDFS : {hdfs_trips.count()}')

Lignes stockees dans HDFS : 213941


In [8]:
# Arreter le consumer HDFS
query.stop()
print('Kafka → HDFS arrete')

# Stats finales
hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
print(f'\nTotal stocke dans HDFS : {hdfs_trips.count()} lignes')
print('Ces donnees sont maintenant disponibles pour le batch (02_silver_to_gold)')

# Liberer les ressources du cluster pour les notebooks suivants
spark.stop()
print('Spark session fermee')

Kafka → HDFS arrete

Total stocke dans HDFS : 213941 lignes
Ces donnees sont maintenant disponibles pour le batch (02_silver_to_gold)
Spark session fermee
